# Serial correlation: FGLS of Perron & González-Coya (2026)

We simulate the design of [PG26] Table 1: AR(1) errors with $\rho = 0.8$ and a regressor $w_t = v_t + e_{t-1}$ that reacts to past shocks (pre-determined but not exogenous). OLS is inconsistent here; FGLS based on the Durbin regression is not.

In [ ]:
import warnings

import numpy as np
import pandas as pd

import pyfgls as pf

warnings.simplefilter("ignore", pf.exceptions.PyFGLSWarning)


In [ ]:
from pyfgls.simulate import serial_dgp

sim = serial_dgp(500, random_state=0)
idx = pd.period_range('1980-01', periods=500, freq='M')
y = pd.Series(sim.y, index=idx, name='y')
X = pd.DataFrame(sim.X, index=idx, columns=['const', 'w'])
res = pf.SerialFGLS(y, X).fit()
print(res.summary())

The BIC path over AR orders (all orders estimated on a common sample):

In [ ]:
res.bic_path.round(3)

Compare with OLS (HAC standard errors) and Cochrane–Orcutt, which estimates the filter from the (inconsistent) OLS residuals:

In [ ]:
co = pf.benchmarks.cochrane_orcutt(y, X)
pf.compare([pf.benchmarks.ols(y, X), co, res]).round(4)

A small Monte Carlo makes the point (the paper uses 10,000 replications):

In [ ]:
from pyfgls.serial import estimate_serial
from pyfgls._linalg import lstsq

rng = np.random.default_rng(1)
err = {'OLS': [], 'Cochrane-Orcutt': [], 'FGLS': []}
for _ in range(200):
    s = serial_dgp(500, random_state=rng)
    err['OLS'].append(lstsq(s.X, s.y)[1] - 1)
    err['Cochrane-Orcutt'].append(pf.benchmarks.cochrane_orcutt(s.y, s.X).params.iloc[1] - 1)
    err['FGLS'].append(estimate_serial(s.y, s.X, ['c', 'w'], kmax=12).beta[1] - 1)
pd.DataFrame({k: {'bias': np.mean(v), 'RMSE': np.sqrt(np.mean(np.square(v)))}
              for k, v in err.items()}).T.round(3)

In [ ]:
res.plot_diagnostics();